In [2]:
#%pip install openpyxl
%pip install ecos

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for ecos: filename=ecos-2.0.14-cp314-cp314-win_amd64.whl size=67526 sha256=f2b927df4f8c838eb016ded935648808f63f152410c7ff5be0b1b2cf0a92321e
  Stored in directory: c:\users\01834179173\appdata\local\pip\cache\wheels\a3\3a\b7\9116747930ffa38c7b4a453f8a4765cfd9ae33e5c841075a39
Successfully built ecos
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import numpy as np
import cvxpy as cp
from hmmlearn.hmm import GaussianHMM
from tqdm import tqdm
import os
import warnings

# Silenciar avisos
warnings.filterwarnings('ignore')

# ==============================================================================
# MÓDULO 1: O MOTOR DE OTIMIZAÇÃO (HMM + EWMA + L1)
# ==============================================================================
def otimizacao_condicional_hmm_ewma(retornos_historicos, pesos_atuais, custo_transacional=0.02, aversao_risco=1.0):
    
    # PASSO 1: DETECÇÃO DO REGIME DE MERCADO VIA HMM
    retornos_mercado = retornos_historicos.mean(axis=1).values.reshape(-1, 1)
    
    modelo_hmm = GaussianHMM(n_components=2, covariance_type="full", random_state=42)
    modelo_hmm.fit(retornos_mercado)
    estado_atual = modelo_hmm.predict(retornos_mercado)[-1]
    
    # PASSO 2: CÁLCULO EWMA MODULADO PELO REGIME
    alpha_dinamico = 0.04 if estado_atual == 0 else 0.10
        
    ewma_dados = retornos_historicos.ewm(alpha=alpha_dinamico)
    vetor_retornos_esp = ewma_dados.mean().iloc[-1].values
    
    matriz_cov_ewma = ewma_dados.cov().iloc[-len(retornos_historicos.columns):].values
    
    # PASSO 3: OTIMIZAÇÃO CONVEXA COM PENALIDADE L1
    n_ativos = len(retornos_historicos.columns)
    w = cp.Variable(n_ativos)
    
    matriz_cov_ewma = matriz_cov_ewma + np.eye(n_ativos) * 1e-9
    
    risco = cp.quad_form(w, cp.psd_wrap(matriz_cov_ewma))
    retorno = vetor_retornos_esp.T @ w
    
    giro_l1 = cp.norm(w - pesos_atuais, 1)
    custo_total = custo_transacional * giro_l1
    
    funcao_objetivo = cp.Maximize(retorno - (aversao_risco * risco) - custo_total)
    restricoes = [cp.sum(w) == 1, w >= 0]
    
    problema = cp.Problem(funcao_objetivo, restricoes)
    problema.solve(solver=cp.ECOS) 
    
    if w.value is None:
        raise ValueError("Otimizador não encontrou solução convexa.")
        
    return w.value

# ==============================================================================
# MÓDULO 2: O LAÇO DO BACKTEST (JANELAS MÓVEIS)
# ==============================================================================
def executar_backtest_dinamico(diretorio_dados, custo_transacional_pct=0.02, aversao_risco=1.0, meses_treino_inicial=60):
    print("=== INÍCIO DO BACKTEST: JANELAS MÓVEIS (HMM + EWMA + L1) ===")
    
    # 1. Carregar Retornos
    caminho_retornos = os.path.join(diretorio_dados, "matriz_retornos_simples_v2.csv")
    print("1. A carregar a matriz de retornos contínua...")
    df_retornos = pd.read_csv(caminho_retornos, index_col='Data', parse_dates=True)
    
    colunas_ativos = [col for col in df_retornos.columns if col != 'IBOV']
    df_ativos = df_retornos[colunas_ativos]
    n_ativos = len(colunas_ativos)
    
    # 2. Carregar CDI e SELIC Externos
    print("2. A carregar taxas CDI e SELIC de fontes externas...")
    caminho_cdi = os.path.join(diretorio_dados, "CDI_2010_2026.xlsx")
    caminho_selic = os.path.join(diretorio_dados, "SELIC_2010_2026.xlsx")
    
    df_cdi = pd.read_excel(caminho_cdi)
    df_cdi = df_cdi.rename(columns={'Date': 'Data', 'valor': 'CDI'}).set_index('Data')
    df_cdi.index = pd.to_datetime(df_cdi.index)
    
    df_selic = pd.read_excel(caminho_selic)
    df_selic = df_selic.rename(columns={'Date': 'Data', 'SELIC': 'SELIC'}).set_index('Data')
    df_selic.index = pd.to_datetime(df_selic.index)
    
    serie_cdi = df_cdi['CDI'].reindex(df_ativos.index).ffill()
    
    # Excesso de Retorno (Prémio de Risco) para passar ao HMM
    df_excesso = df_ativos.sub(serie_cdi, axis=0)
    
    print("3. A configurar pontos de rebalanceamento mensais...")
    datas_rebalanceamento = df_excesso.groupby([df_excesso.index.year, df_excesso.index.month]).apply(lambda x: x.index[-1]).values
    
    indice_inicio_backtest = meses_treino_inicial
    datas_backtest = datas_rebalanceamento[indice_inicio_backtest:]
    
    pesos_atuais = np.array([1.0 / n_ativos] * n_ativos)
    
    historico_pesos = pd.DataFrame(index=datas_backtest, columns=colunas_ativos, dtype=float)
    historico_turnover = pd.Series(index=datas_backtest, dtype=float)
    
    print(f"4. A iniciar a máquina do tempo: {len(datas_backtest)} rebalanceamentos a processar...")
    
    for data_corte in tqdm(datas_backtest):
        # O HMM e o Otimizador analisam o Excesso de Retorno
        dados_janela = df_excesso.loc[:data_corte]
        
        try:
            novos_pesos = otimizacao_condicional_hmm_ewma(
                retornos_historicos=dados_janela,
                pesos_atuais=pesos_atuais,
                custo_transacional=custo_transacional_pct,
                aversao_risco=aversao_risco
            )
            
            novos_pesos = np.where(novos_pesos < 1e-4, 0, novos_pesos)
            novos_pesos /= np.sum(novos_pesos) 
            
        except Exception as e:
            data_formatada = pd.to_datetime(data_corte).date()
            print(f"\n   -> Aviso na data {data_formatada}: Solução não convergiu. Mantendo pesos. ({e})")
            novos_pesos = pesos_atuais.copy()
            
        turnover_mensal = np.sum(np.abs(novos_pesos - pesos_atuais)) / 2.0
        
        historico_pesos.loc[data_corte] = novos_pesos
        historico_turnover.loc[data_corte] = turnover_mensal
        
        pesos_atuais = novos_pesos.copy()

    print("\n5. A finalizar o Backtest e a exportar as alocações dinâmicas...")
    caminho_historico = os.path.join(diretorio_dados, "Historico_Alocacao_HMM_EWMA_Maximo_Sharpe.csv")
    caminho_turnover = os.path.join(diretorio_dados, "Historico_Turnover_Maximo_Sharpe.csv")
    
    historico_pesos.to_csv(caminho_historico)
    historico_turnover.to_csv(caminho_turnover)
    
    print("=== RESUMO DA SIMULAÇÃO - Maximo Sharpe - Aversao a Risco = 1.0 ===")
    print(f"Total de meses rebalanceados: {len(historico_pesos)}")
    print(f"Turnover médio mensal da carteira: {historico_turnover.mean() * 100:.2f}%")
    print("===========================================================")
    
    print("6. A formatar o Painel de Evolução por Ação (Long Data)...")
    df_painel = historico_pesos.reset_index().melt(
        id_vars='index', 
        var_name='Ativo', 
        value_name='Peso'
    )
    df_painel.rename(columns={'index': 'Data'}, inplace=True)
    df_painel = df_painel[df_painel['Peso'] > 0.0001].sort_values(by=['Data', 'Peso'], ascending=[True, False])
    
    caminho_painel = os.path.join(diretorio_dados, "painel_alocacao_mensal_maximo_sharpe.csv")
    df_painel.to_csv(caminho_painel, index=False)
    print(f"   -> Painel de visualização salvo em: {caminho_painel}")

    return historico_pesos, historico_turnover

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"

pesos_no_tempo, giro_carteira = executar_backtest_dinamico(
    diretorio_dados=pasta_dados, 
    custo_transacional_pct=0.005, 
    aversao_risco=1.0, 
    meses_treino_inicial=60
)

=== INÍCIO DO BACKTEST: JANELAS MÓVEIS (HMM + EWMA + L1) ===
1. A carregar a matriz de retornos contínua...
2. A carregar taxas CDI e SELIC de fontes externas...
3. A configurar pontos de rebalanceamento mensais...
4. A iniciar a máquina do tempo: 132 rebalanceamentos a processar...


 44%|████▍     | 58/132 [03:14<04:07,  3.35s/it]


KeyboardInterrupt: 

=== INÍCIO DO BACKTEST: JANELAS MÓVEIS (HMM + EWMA + L1) ===
1. A carregar a matriz de retornos contínua...
2. A configurar pontos de rebalanceamento mensais...
3. A iniciar a máquina do tempo: 132 rebalanceamentos a processar...
100%|██████████| 132/132 [07:09<00:00,  3.26s/it]

4. A finalizar o Backtest e a exportar as alocações dinâmicas...
=== RESUMO DA SIMULAÇÃO ===
Total de meses rebalanceados: 132
Turnover médio mensal da carteira: 0.51%
===========================================================
5. A formatar o Painel de Evolução por Ação (Long Data)...
   -> Painel de visualização salvo em: C:\VSCodeWorkspace\TCC_Escrito\data\painel_alocacao_mensal.csv

In [4]:
import pandas as pd
import numpy as np
import cvxpy as cp
import os

def teste_integridade_dados_min_var(diretorio_dados):
    print("=== TESTE DE INTEGRIDADE: MÍNIMA VARIÂNCIA (1 MÊS) ===")
    
    # 1. Carregar Dados
    caminho_retornos = os.path.join(diretorio_dados, "matriz_retornos_simples_v2.csv")
    if not os.path.exists(caminho_retornos):
        return "ERRO: Ficheiro de retornos não encontrado."
        
    df_retornos = pd.read_csv(caminho_retornos, index_col='Data', parse_dates=True)
    df_ativos = df_retornos.drop(columns=['IBOV'], errors='ignore')
    
    # Pegar apenas nos últimos 60 meses para o teste
    dados_teste = df_ativos.iloc[-60:]
    n_ativos = len(dados_teste.columns)
    
    # 2. Verificações de Integridade
    print(f"-> Ativos encontrados: {n_ativos}")
    print(f"-> Dados nulos (NaNs) na amostra: {dados_teste.isna().sum().sum()}")
    
    # 3. Gerar Covariância (Simulação EWMA Simples)
    matriz_cov = dados_teste.ewm(alpha=0.04).cov().iloc[-n_ativos:].values
    
    print(f"-> Valores Infinitos na Covariância: {np.isinf(matriz_cov).sum()}")
    
    # 4. Teste de Solver CVXPY (Mínima Variância)
    print("\n-> A testar o otimizador ECOS (Mínima Variância)...")
    w = cp.Variable(n_ativos)
    pesos_atuais = np.array([1.0 / n_ativos] * n_ativos)
    
    # Truque de estabilidade numérica
    matriz_cov = matriz_cov + np.eye(n_ativos) * 1e-9
    
    risco = cp.quad_form(w, cp.psd_wrap(matriz_cov))
    giro_l1 = cp.norm(w - pesos_atuais, 1)
    custo = 0.02 * giro_l1
    
    # FUNÇÃO OBJETIVO PURA DE MÍNIMA VARIÂNCIA
    obj = cp.Minimize(risco + custo)
    restricoes = [cp.sum(w) == 1, w >= 0]
    
    problema = cp.Problem(obj, restricoes)
    
    try:
        problema.solve(solver=cp.ECOS)
        if w.value is None:
            print("❌ ERRO: Otimizador falhou. Solução não convexa.")
        else:
            print("✅ SUCESSO: Otimizador ECOS resolveu a equação!")
            soma_pesos = np.sum(w.value)
            print(f"-> Soma total dos pesos: {soma_pesos:.4f} (Deve ser 1.0000)")
            
            # Limpeza e Top 5
            pesos = np.where(w.value < 1e-4, 0, w.value)
            pesos /= np.sum(pesos)
            top_ativos = pd.Series(pesos, index=dados_teste.columns).sort_values(ascending=False).head(5)
            
            print("\n🏆 Top 5 Ativos de Mínima Variância neste teste:")
            for ativo, peso in top_ativos.items():
                print(f"  - {ativo}: {peso:.2%}")
                
    except Exception as e:
        print(f"❌ ERRO CRÍTICO NO SOLVER: {e}")

# Executar
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"
teste_integridade_dados_min_var(pasta_dados)

=== TESTE DE INTEGRIDADE: MÍNIMA VARIÂNCIA (1 MÊS) ===
-> Ativos encontrados: 136
-> Dados nulos (NaNs) na amostra: 0
-> Valores Infinitos na Covariância: 0

-> A testar o otimizador ECOS (Mínima Variância)...
✅ SUCESSO: Otimizador ECOS resolveu a equação!
-> Soma total dos pesos: 1.0000 (Deve ser 1.0000)

🏆 Top 5 Ativos de Mínima Variância neste teste:
  - RDNI3: 0.74%
  - NEXP3: 0.74%
  - INEP4: 0.74%
  - FICT3: 0.74%
  - EMBJ3: 0.74%


In [7]:
%pip install osqp

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import cvxpy as cp
from hmmlearn.hmm import GaussianHMM
from tqdm import tqdm
import os
import warnings

# Silenciar avisos
warnings.filterwarnings('ignore')

# ==============================================================================
# MÓDULO 1: O MOTOR DE OTIMIZAÇÃO (HMM + EWMA + L1)
# ==============================================================================
def otimizacao_condicional_hmm_ewma(retornos_historicos, pesos_atuais, custo_transacional=0.02, aversao_risco=1.0):
    
    # PASSO 1: DETECÇÃO DO REGIME DE MERCADO VIA HMM
    retornos_mercado = retornos_historicos.mean(axis=1).values.reshape(-1, 1)
    
    modelo_hmm = GaussianHMM(n_components=2, covariance_type="full", random_state=42)
    modelo_hmm.fit(retornos_mercado)
    estado_atual = modelo_hmm.predict(retornos_mercado)[-1]
    
    # PASSO 2: CÁLCULO EWMA MODULADO PELO REGIME
    alpha_dinamico = 0.04 if estado_atual == 0 else 0.10
        
    ewma_dados = retornos_historicos.ewm(alpha=alpha_dinamico)
    vetor_retornos_esp = ewma_dados.mean().iloc[-1].values
    
    matriz_cov_ewma = ewma_dados.cov().iloc[-len(retornos_historicos.columns):].values
    
    # PASSO 3: OTIMIZAÇÃO CONVEXA COM PENALIDADE L1
    n_ativos = len(retornos_historicos.columns)
    w = cp.Variable(n_ativos)
    
    matriz_cov_ewma = matriz_cov_ewma + np.eye(n_ativos) * 1e-9
    
    risco = cp.quad_form(w, cp.psd_wrap(matriz_cov_ewma))
    retorno = vetor_retornos_esp.T @ w
    
    giro_l1 = cp.norm(w - pesos_atuais, 1)
    custo_total = custo_transacional * giro_l1
    
    funcao_objetivo = cp.Maximize(retorno - (aversao_risco * risco) - custo_total)
    restricoes = [cp.sum(w) == 1, w >= 0]
    
    problema = cp.Problem(funcao_objetivo, restricoes)
    #problema.solve(solver=cp.ECOS) 
    problema.solve(solver=cp.OSQP) 
    
    if w.value is None:
        raise ValueError("Otimizador não encontrou solução convexa.")
        
    return w.value

# ==============================================================================
# MÓDULO 2: O LAÇO DO BACKTEST (JANELAS MÓVEIS)
# ==============================================================================
def executar_backtest_dinamico(diretorio_dados, custo_transacional_pct=0.02, aversao_risco=1.0, meses_treino_inicial=60):
    print("=== INÍCIO DO BACKTEST: JANELAS MÓVEIS (HMM + EWMA + L1) ===")
    
    # 1. Carregar Retornos
    caminho_retornos = os.path.join(diretorio_dados, "matriz_retornos_simples_v2.csv")
    print("1. A carregar a matriz de retornos contínua...")
    df_retornos = pd.read_csv(caminho_retornos, index_col='Data', parse_dates=True)
    
    colunas_ativos = [col for col in df_retornos.columns if col != 'IBOV']
    df_ativos = df_retornos[colunas_ativos]
    n_ativos = len(colunas_ativos)
    
    # 2. Carregar CDI e SELIC Externos
    print("2. A carregar taxas CDI e SELIC de fontes externas...")
    caminho_cdi = os.path.join(diretorio_dados, "CDI_2010_2026.xlsx")
    caminho_selic = os.path.join(diretorio_dados, "SELIC_2010_2026.xlsx")
    
    df_cdi = pd.read_excel(caminho_cdi)
    df_cdi = df_cdi.rename(columns={'Date': 'Data', 'valor': 'CDI'}).set_index('Data')
    df_cdi.index = pd.to_datetime(df_cdi.index)
    
    df_selic = pd.read_excel(caminho_selic)
    df_selic = df_selic.rename(columns={'Date': 'Data', 'SELIC': 'SELIC'}).set_index('Data')
    df_selic.index = pd.to_datetime(df_selic.index)
    
    serie_cdi = df_cdi['CDI'].reindex(df_ativos.index).ffill()
    
    # Excesso de Retorno (Prémio de Risco) para passar ao HMM
    df_excesso = df_ativos.sub(serie_cdi, axis=0)
    
    print("3. A configurar pontos de rebalanceamento mensais...")
    datas_rebalanceamento = df_excesso.groupby([df_excesso.index.year, df_excesso.index.month]).apply(lambda x: x.index[-1]).values
    
    indice_inicio_backtest = meses_treino_inicial
    datas_backtest = datas_rebalanceamento[indice_inicio_backtest:]
    
    pesos_atuais = np.array([1.0 / n_ativos] * n_ativos)
    
    historico_pesos = pd.DataFrame(index=datas_backtest, columns=colunas_ativos, dtype=float)
    historico_turnover = pd.Series(index=datas_backtest, dtype=float)
    
    print(f"4. A iniciar a máquina do tempo: {len(datas_backtest)} rebalanceamentos a processar...")
    
    for data_corte in tqdm(datas_backtest):
        # O HMM e o Otimizador analisam o Excesso de Retorno
        dados_janela = df_excesso.loc[:data_corte]
        
        try:
            novos_pesos = otimizacao_condicional_hmm_ewma(
                retornos_historicos=dados_janela,
                pesos_atuais=pesos_atuais,
                custo_transacional=custo_transacional_pct,
                aversao_risco=aversao_risco
            )
            
            novos_pesos = np.where(novos_pesos < 1e-4, 0, novos_pesos)
            novos_pesos /= np.sum(novos_pesos) 
            
        except Exception as e:
            data_formatada = pd.to_datetime(data_corte).date()
            print(f"\n   -> Aviso na data {data_formatada}: Solução não convergiu. Mantendo pesos. ({e})")
            novos_pesos = pesos_atuais.copy()
            
        turnover_mensal = np.sum(np.abs(novos_pesos - pesos_atuais)) / 2.0
        
        historico_pesos.loc[data_corte] = novos_pesos
        historico_turnover.loc[data_corte] = turnover_mensal
        
        pesos_atuais = novos_pesos.copy()

    print("\n5. A finalizar o Backtest e a exportar as alocações dinâmicas...")
    caminho_historico = os.path.join(diretorio_dados, "Historico_Alocacao_HMM_EWMA_Maximo_Sharpe.csv")
    caminho_turnover = os.path.join(diretorio_dados, "Historico_Turnover_Maximo_Sharpe.csv")
    
    historico_pesos.to_csv(caminho_historico)
    historico_turnover.to_csv(caminho_turnover)
    
    print("=== RESUMO DA SIMULAÇÃO - Maximo Sharpe - Aversao a Risco = 1.0 ===")
    print(f"Total de meses rebalanceados: {len(historico_pesos)}")
    print(f"Turnover médio mensal da carteira: {historico_turnover.mean() * 100:.2f}%")
    print("===========================================================")
    
    print("6. A formatar o Painel de Evolução por Ação (Long Data)...")
    df_painel = historico_pesos.reset_index().melt(
        id_vars='index', 
        var_name='Ativo', 
        value_name='Peso'
    )
    df_painel.rename(columns={'index': 'Data'}, inplace=True)
    df_painel = df_painel[df_painel['Peso'] > 0.0001].sort_values(by=['Data', 'Peso'], ascending=[True, False])
    
    caminho_painel = os.path.join(diretorio_dados, "painel_alocacao_mensal_maximo_sharpe.csv")
    df_painel.to_csv(caminho_painel, index=False)
    print(f"   -> Painel de visualização salvo em: {caminho_painel}")

    return historico_pesos, historico_turnover

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"

pesos_no_tempo, giro_carteira = executar_backtest_dinamico(
    diretorio_dados=pasta_dados, 
    custo_transacional_pct=0.005, 
    aversao_risco=1.0, 
    meses_treino_inicial=60
)

=== INÍCIO DO BACKTEST: JANELAS MÓVEIS (HMM + EWMA + L1) ===
1. A carregar a matriz de retornos contínua...
2. A carregar taxas CDI e SELIC de fontes externas...
3. A configurar pontos de rebalanceamento mensais...
4. A iniciar a máquina do tempo: 132 rebalanceamentos a processar...


100%|██████████| 132/132 [06:43<00:00,  3.05s/it]


5. A finalizar o Backtest e a exportar as alocações dinâmicas...
=== RESUMO DA SIMULAÇÃO - Maximo Sharpe - Aversao a Risco = 1.0 ===
Total de meses rebalanceados: 132
Turnover médio mensal da carteira: 38.01%
6. A formatar o Painel de Evolução por Ação (Long Data)...
   -> Painel de visualização salvo em: C:\VSCodeWorkspace\TCC_Escrito\data\painel_alocacao_mensal_maximo_sharpe.csv
